In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "data" / "plasmidscope_primary").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PP = ROOT / "data" / "plasmidscope_primary"

master = pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False)

# Standing analysis base: drop the two non-habitat artifact buckets (see analysis.ipynb).
real = master[~master["hab_top"].isin(["Simulated-artifact", "Lab-artifact"])].copy()

# Small = under 20 kbp.
SMALL_BP = 20_000
is_small = real["size_bp"] < SMALL_BP
counts = pd.Series({"< 20 kbp": is_small.sum(), ">= 20 kbp": (~is_small).sum()})

print(f"analysis base: {len(real):,} / {len(master):,} plasmids")
for label, n in counts.items():
    print(f"  {label:<10s} {n:>7,}  {n / len(real) * 100:5.2f}%")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.pie(counts, labels=counts.index, colors=["#2a78d6", "#c9d6e5"],
       autopct=lambda p: f"{p:.1f}%\n({p / 100 * counts.sum():,.0f})",
       startangle=90, counterclock=False, textprops={"fontsize": 10})
ax.set_title(f"Small plasmids (< 20 kbp) — {len(real):,} plasmids")
plt.tight_layout()
plt.show()


In [ ]:
# Payload-free ("cryptic") = carries none of the three payload classes.
# NOTE: conjugation machinery is NOT disqualifying here, unlike the locked definition in
# reports/small_cryptic_methodology.md. The 71 plasmids PlasAnn could not annotate are
# filled with 0, i.e. counted as payload-free (same handling as the report).
PAYLOAD = ["card_n_arg", "plasann_n_virulence", "plasann_n_metal_biocide"]
is_cryptic = (real[PAYLOAD].fillna(0) == 0).all(axis=1)

small = real[is_small]
small_cryptic = is_cryptic[is_small]

print(f"small (< {SMALL_BP // 1000} kbp):        {len(small):>7,}")
print(f"  payload-free (cryptic): {small_cryptic.sum():>7,}  "
      f"{small_cryptic.mean() * 100:5.2f}% of small  "
      f"({small_cryptic.sum() / len(real) * 100:.2f}% of all {len(real):,})")
print(f"  carries payload:        {(~small_cryptic).sum():>7,}  "
      f"{(~small_cryptic).mean() * 100:5.2f}% of small")


In [ ]:
# Small plasmids carrying PlasAnn conjugation-category genes, stratified by the mob_typer call.
# PlasAnn's "conjugation" category is broad (relaxase/MOB genes count), so the mob_typer stratum is
# what separates genuinely self-transmissible plasmids from merely mobilizable ones.
STRATA = ["conjugative", "mobilizable", "non-mobilizable"]
sc = real[is_small & (real["plasann_n_conjugation"].fillna(0) > 0)].copy()
sc["stratum"] = pd.Categorical(sc["mob_mobility"], STRATA)

g = sc.groupby("stratum", observed=False)
machinery = pd.DataFrame({
    "n": g.size(),
    "% of small": g.size() / is_small.sum() * 100,
    "med conj genes": g["plasann_n_conjugation"].median(),
    "% relaxase": g["mob_relaxase"].apply(lambda x: x.notna().mean() * 100),
    "% MPF": g["mob_mpf"].apply(lambda x: x.notna().mean() * 100),
    "% oriT (mob)": g["mob_orit"].apply(lambda x: x.notna().mean() * 100),
    "% oriT (PlasAnn)": g["plasann_has_orit"].apply(lambda x: (x == 1).mean() * 100),
})
machinery.loc["ALL"] = [
    len(sc), len(sc) / is_small.sum() * 100, sc["plasann_n_conjugation"].median(),
    sc["mob_relaxase"].notna().mean() * 100, sc["mob_mpf"].notna().mean() * 100,
    sc["mob_orit"].notna().mean() * 100, (sc["plasann_has_orit"] == 1).mean() * 100,
]

print(f"small (< {SMALL_BP // 1000} kbp) with >=1 PlasAnn conjugation gene: "
      f"{len(sc):,} / {is_small.sum():,} ({len(sc) / is_small.sum() * 100:.2f}%)\n")
print(machinery.round(1).to_string())

# Which relaxase family, per stratum (primary family only; mob_families is ';'-separated).
fam = sc["mob_families"].str.split(";").str[0].fillna("(none)")
print("\nprimary MOB family x stratum (% of stratum):")
print(pd.crosstab(fam, sc["stratum"], normalize="columns").mul(100).round(1)
        .loc[lambda d: d.max(axis=1) >= 1].to_string())

print(f"\nMPF type among the {sc['mob_mpf'].notna().sum()} MPF-positive:")
print(sc["mob_mpf"].value_counts().to_string())


In [ ]:
import numpy as np

# Palette: slots 1-4 of the validated categorical default + a neutral for "absent".
S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#dcdbd4"

def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9, 10.5))
y = np.arange(len(STRATA))

# A — how the 22,975 split across mob_typer calls (one series -> one colour, no legend).
n = machinery.loc[STRATA, "n"]
ax1.barh(y, n, height=0.34, color=S1)
for i, v in enumerate(n):
    ax1.text(v + n.max() * 0.012, i, f"{v:,.0f}", va="center", fontsize=9, color=INK2)
ax1.set(yticks=y, yticklabels=STRATA, xlim=(0, n.max() * 1.12))
ax1.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax1.invert_yaxis()
ax1.set_xlabel("plasmids", fontsize=9, color=INK2)
ax1.set_title(f"Small (< {SMALL_BP // 1000} kb) plasmids with a PlasAnn conjugation gene, "
              f"by mob_typer call  (n = {len(sc):,})", fontsize=11, color=INK, loc="left")
tidy(ax1)

# B — which components of the apparatus are actually detected (all three from mob_typer).
comps = [("relaxase", "% relaxase", S1), ("MPF", "% MPF", S2), ("oriT", "% oriT (mob)", S3)]
x, w = np.arange(len(STRATA)), 0.20
for k, (label, col, colour) in enumerate(comps):
    vals = machinery.loc[STRATA, col]
    ax2.bar(x + (k - 1) * w, vals, width=w * 0.85, color=colour, label=label)
    for xi, v in zip(x + (k - 1) * w, vals):
        ax2.text(xi, v + 2, f"{v:.1f}", ha="center", fontsize=8, color=INK2)
ax2.set(xticks=x, xticklabels=STRATA, yticks=[], ylim=(0, 118))
ax2.set_title("Conjugation-machinery components detected (% of stratum)",
              fontsize=11, color=INK, loc="left")
ax2.legend(frameon=False, fontsize=9, labelcolor=INK2, ncol=3, loc="upper right")
tidy(ax2)

# C — relaxase (MOB) family composition; grey = no family called at all.
FAMS = ["MOBP", "MOBQ", "MOBV"]
fam1 = sc["mob_families"].str.split(";").str[0]
famgrp = np.where(fam1.isna(), "no family called",
                  np.where(fam1.isin(FAMS), fam1, "other MOB"))
order = FAMS + ["other MOB", "no family called"]
comp = (pd.crosstab(sc["stratum"], famgrp, normalize="index")
          .mul(100).reindex(columns=order, fill_value=0))

left = np.zeros(len(STRATA))
for col, colour, lab_ink in zip(order, [S1, S2, S3, S4, NEUT],
                                ["white", "white", "white", INK, INK]):
    vals = comp.loc[STRATA, col].values
    ax3.barh(y, vals, left=left, height=0.34, color=colour, label=col,
             edgecolor="white", linewidth=1.5)          # white = the 2px surface gap
    for i, (v, l) in enumerate(zip(vals, left)):
        if v >= 8:                                       # only label segments wide enough to fit
            ax3.text(l + v / 2, i, f"{v:.0f}", ha="center", va="center",
                     fontsize=8, color=lab_ink)
    left += vals
ax3.set(yticks=y, yticklabels=STRATA, xlim=(0, 100), xticks=[0, 25, 50, 75, 100])
ax3.invert_yaxis()
ax3.set_xlabel("% of stratum", fontsize=9, color=INK2)
ax3.set_title("Relaxase (MOB) family composition", fontsize=11, color=INK, loc="left")
ax3.legend(frameon=False, fontsize=9, labelcolor=INK2, ncol=5,
           loc="upper center", bbox_to_anchor=(0.5, -0.22))
tidy(ax3)

plt.tight_layout()
plt.show()


In [ ]:
import glob

# Per-gene PlasAnn rows for the 300 mob_typer-conjugative small plasmids.
# Glob *shard_* — there are TWO shard series (shard_* primary, mshard_* reshard); globbing only
# one silently drops ~27% of the data (pitfall recorded in reports/small_cryptic_methodology.md).
# ~60 s over 1,072 files.
conj_ids = set(sc.loc[sc["stratum"] == "conjugative", "plasmid_id"])
parts = []
for f in sorted(glob.glob(str(ROOT / "data" / "plasann_run" / "annot" / "*shard_*.tsv.gz"))):
    d = pd.read_csv(f, sep="\t", low_memory=False)
    d = d[d["plasmid_id"].isin(conj_ids)]
    if len(d):
        parts.append(d)
genes = pd.concat(parts, ignore_index=True)

n_cov = genes["plasmid_id"].nunique()
print(f"per-gene coverage: {n_cov}/{len(conj_ids)} conjugative plasmids, {len(genes):,} CDS rows\n")

# What functional classes are present, and in how many of the plasmids.
vc = genes["Category"].value_counts()
cat = pd.DataFrame({
    "CDS": vc,
    "% of CDS": vc / len(genes) * 100,
    "plasmids": genes.groupby("Category")["plasmid_id"].nunique(),
})
cat["% of plasmids"] = cat["plasmids"] / n_cov * 100
print(cat.sort_values("plasmids", ascending=False).round(1).to_string())

# How much of the coding capacity is unannotatable (PlasAnn labels those CDS "ORF").
is_orf = genes["Gene Name"].eq("ORF")
dark = is_orf.groupby(genes["plasmid_id"]).mean()
print(f"\nunannotatable (ORF) CDS: {is_orf.mean() * 100:.1f}% pooled | "
      f"median per-plasmid dark fraction {dark.median():.2f} | "
      f"{(dark == 1).sum()} plasmids ({(dark == 1).mean() * 100:.1f}%) entirely dark")

# The actual named genes, excluding both ORFs and the conjugation machinery itself.
other = genes[~is_orf & genes["Category"].ne("Conjugation")]
top = (other.groupby(["Category", "Gene Name"])
            .agg(CDS=("plasmid_id", "size"), plasmids=("plasmid_id", "nunique"))
            .sort_values("plasmids", ascending=False).head(25))
print(f"\ntop non-conjugation genes ({other['Gene Name'].nunique():,} distinct names, "
      f"{len(other):,} CDS):")
print(top.to_string())


In [ ]:
from scipy import stats

sizes = {s: sc.loc[sc["stratum"] == s, "size_bp"].values for s in STRATA}

print("size_bp by mobilization category (within the conjugation-gene-positive small set):")
print(sc.groupby("stratum", observed=False)["size_bp"]
        .describe()[["count", "min", "25%", "50%", "75%", "max", "mean"]].round(0).to_string())

H, p = stats.kruskal(*sizes.values())
print(f"\nKruskal-Wallis across the three categories: H = {H:,.1f}, p = {p:.3g}")

rows = []
for a, b in [("conjugative", "mobilizable"), ("conjugative", "non-mobilizable"),
             ("mobilizable", "non-mobilizable")]:
    u, pu = stats.mannwhitneyu(sizes[a], sizes[b])
    rows.append({"pair": f"{a} vs {b}",
                 "median A": np.median(sizes[a]), "median B": np.median(sizes[b]),
                 "rank-biserial": 2 * u / (len(sizes[a]) * len(sizes[b])) - 1, "p": pu})
pw = pd.DataFrame(rows)
pw["p (BH)"] = stats.false_discovery_control(pw["p"])
print("\npairwise Mann-Whitney (BH-corrected):")
print(pw.round({"median A": 0, "median B": 0, "rank-biserial": 3}).to_string(index=False))

# One series -> one colour; the boxes are categories, not series, so no legend.
fig, ax = plt.subplots(figsize=(9, 3.6))
y = np.arange(len(STRATA))
ax.boxplot([sizes[s] for s in STRATA], positions=y, vert=False, widths=0.34, showfliers=False,
           patch_artist=True,
           boxprops=dict(facecolor=S1, edgecolor="none"),
           medianprops=dict(color="white", linewidth=1.5),
           whiskerprops=dict(color=INK2, linewidth=1),
           capprops=dict(color=INK2, linewidth=1))
for i, s in enumerate(STRATA):
    med = np.median(sizes[s])
    ax.text(med + 400, i - 0.28, f"{med:,.0f}", fontsize=8, color=INK2, va="center")
ax.axvline(SMALL_BP, color=GRID, linewidth=1)
ax.text(SMALL_BP - 300, 0.04, f"{SMALL_BP // 1000} kb cut", fontsize=8, color=INK2,
        ha="right", transform=ax.get_xaxis_transform())
ax.set(yticks=y, yticklabels=STRATA, xlim=(0, SMALL_BP * 1.06))
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.invert_yaxis()
ax.set_xlabel("size (bp)", fontsize=9, color=INK2)
ax.set_title("Plasmid length by mobilization category  (whiskers 1.5×IQR, outliers hidden)",
             fontsize=11, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()


In [ ]:
# Figure for cell 5: what the 300 mob_typer-conjugative small plasmids actually encode.
catf = cat.sort_values("% of plasmids")                      # ascending -> largest ends up on top
# Gene names recur across categories (e.g. reP), so count by name alone, not by (category, name).
topg = (other.groupby("Gene Name")["plasmid_id"].nunique()
             .sort_values(ascending=False).head(20).iloc[::-1])

fig, (axA, axB) = plt.subplots(2, 1, figsize=(9, 12),
                               gridspec_kw={"height_ratios": [len(catf), len(topg)]})

# A — prevalence vs coding share. Two series, both percentages, one axis.
yA, h = np.arange(len(catf)), 0.36
axA.barh(yA + h / 2, catf["% of plasmids"], height=h * 0.9, color=S1, label="% of plasmids")
axA.barh(yA - h / 2, catf["% of CDS"], height=h * 0.9, color=S2, label="% of CDS")
axA.set(yticks=yA, yticklabels=catf.index, xlim=(0, 100), xticks=[0, 25, 50, 75, 100])
axA.set_xlabel("percent", fontsize=9, color=INK2)
axA.grid(axis="x", color=GRID, linewidth=1)
axA.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="lower right")
axA.set_title(f"What is encoded on the {len(conj_ids)} conjugative small plasmids\n"
              f"{n_cov} with per-gene data · {len(genes):,} CDS · "
              f"{is_orf.mean() * 100:.0f}% of coding capacity unannotatable",
              fontsize=11, color=INK, loc="left")
tidy(axA)

# B — the named genes behind those categories, conjugation machinery excluded.
yB = np.arange(len(topg))
axB.barh(yB, topg.values, height=0.4, color=S1)
for i, v in enumerate(topg.values):
    axB.text(v + topg.max() * 0.015, i, f"{v}", va="center", fontsize=8, color=INK2)
axB.set(yticks=yB, yticklabels=topg.index, xlim=(0, topg.max() * 1.1))
axB.set_xlabel(f"plasmids carrying the gene (of {n_cov})", fontsize=9, color=INK2)
axB.set_title(f"Top non-conjugation genes  ({other['Gene Name'].nunique():,} distinct names)",
              fontsize=11, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


In [ ]:
# Host of the 300 conjugative small plasmids. Two different signals, very different coverage:
#   organism       — observed host string from working_set.tsv, the only host-taxon label for
#                    plasmids without a PLSDB record. Sparse.
#   mob_host_range — mob_typer's PREDICTED host range (a neighbourhood-based call, not an
#                    observation). Complete, but a prediction.
ws = pd.read_csv(PP / "working_set.tsv", sep="\t", low_memory=False,
                 usecols=["plasmid_id", "organism"])
conj = sc[sc["stratum"] == "conjugative"].copy()
conj["organism"] = conj["plasmid_id"].map(dict(zip(ws["plasmid_id"], ws["organism"])))

genus = conj["organism"].str.split().str[0]              # consolidates pv./var. strain suffixes
topA = genus.value_counts().head(12).iloc[::-1]
topB = conj["mob_host_range"].value_counts().head(12).iloc[::-1]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 5))
for ax, top, n_obs, title in [
    (axA, topA, genus.notna().sum(), "Observed host genus (working_set `organism`)"),
    (axB, topB, conj["mob_host_range"].notna().sum(), "Predicted host range (mob_typer)"),
]:
    y = np.arange(len(top))
    ax.barh(y, top.values, height=0.4, color=S1)
    for i, v in enumerate(top.values):
        ax.text(v + top.max() * 0.02, i, f"{v}", va="center", fontsize=8, color=INK2)
    ax.set(yticks=y, yticklabels=top.index, xlim=(0, top.max() * 1.12))
    ax.set_xlabel("plasmids", fontsize=9, color=INK2)
    ax.set_title(f"{title}\n{n_obs}/{len(conj)} with a value ({n_obs / len(conj) * 100:.0f}%)",
                 fontsize=10, color=INK, loc="left")
    tidy(ax)

plt.tight_layout()
plt.show()


In [ ]:
# Habitat and One Health compartment for the 300 conjugative small plasmids.
# Compartments are built from hab_sub + is_clinical, never hab_top — map copied verbatim from
# scripts/analyze_amr_onehealth_mobility.py so this stays consistent with the Phase-3 analysis.
HUMAN_SUBS = {
    "Human: unspecified", "Human: gut/faeces", "Human: blood", "Human: urine",
    "Human: respiratory", "Human: skin/wound", "Human: other clinical", "Human: oral",
}
COMPARTMENT_MAP = {
    "Livestock: pig": "Animal - livestock & poultry",
    "Livestock: cattle": "Animal - livestock & poultry",
    "Livestock: other": "Animal - livestock & poultry",
    "Poultry/bird": "Animal - livestock & poultry",
    "Companion animal": "Animal - companion & aquaculture",
    "Fish/aquaculture": "Animal - companion & aquaculture",
    "Other mammal": "Animal - wildlife & other",
    "Rodent": "Animal - wildlife & other",
    "Insect/arthropod": "Animal - wildlife & other",
    "Other invertebrate": "Animal - wildlife & other",
    "Gut/faeces: unspecified host": "Animal - wildlife & other",
    "Wastewater/sewage": "Environment - engineered interface",
    "Bioreactor": "Environment - engineered interface",
    "Built environment": "Environment - engineered interface",
    "Solid waste/compost": "Environment - engineered interface",
    "Industrial/remediation": "Environment - engineered interface",
    "Terrestrial/soil": "Environment - natural",
    "Aquatic: freshwater/other": "Environment - natural",
    "Aquatic: marine": "Environment - natural",
    "Air": "Environment - natural",
    "Extreme/other": "Environment - natural",
    "Environmental: unspecified": "Environment - natural",
    "Plant": "Plant & food",
    "Food/fermentation": "Plant & food",
    "Algae": "Plant & food",
    "Fungi": "Plant & food",
}
OH_ORDER = ["Human - clinical", "Human - community",
            "Animal - livestock & poultry", "Animal - companion & aquaculture",
            "Animal - wildlife & other",
            "Environment - engineered interface", "Environment - natural", "Plant & food"]

def assign_compartment(row):
    if row["hab_sub"] in HUMAN_SUBS:
        return "Human - clinical" if row["is_clinical"] == 1 else "Human - community"
    return COMPARTMENT_MAP.get(row["hab_sub"])

conj["compartment"] = conj.apply(assign_compartment, axis=1)
n_assigned = conj["compartment"].notna().sum()

topH = conj["hab_sub"].value_counts().head(12).iloc[::-1]
ohc = (conj["compartment"].value_counts().reindex(OH_ORDER, fill_value=0))
ohc["(no compartment)"] = len(conj) - n_assigned
ohc = ohc.iloc[::-1]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))

y = np.arange(len(topH))
axA.barh(y, topH.values, height=0.4, color=S1)
for i, v in enumerate(topH.values):
    axA.text(v + topH.max() * 0.02, i, f"{v}", va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=topH.index, xlim=(0, topH.max() * 1.12))
axA.set_xlabel("plasmids", fontsize=9, color=INK2)
axA.set_title(f"Habitat (hab_sub) — top 12 of {conj['hab_sub'].nunique()}",
              fontsize=10, color=INK, loc="left")
tidy(axA)

# Grey marks the absence of an assignment, not another compartment.
y = np.arange(len(ohc))
axB.barh(y, ohc.values, height=0.4,
         color=[NEUT if k == "(no compartment)" else S1 for k in ohc.index])
for i, v in enumerate(ohc.values):
    axB.text(v + ohc.max() * 0.02, i, f"{v}", va="center", fontsize=8, color=INK2)
axB.set(yticks=y, yticklabels=ohc.index, xlim=(0, ohc.max() * 1.12))
axB.set_xlabel("plasmids", fontsize=9, color=INK2)
axB.set_title(f"One Health compartment (hab_sub + is_clinical)\n"
              f"{n_assigned}/{len(conj)} assigned ({n_assigned / len(conj) * 100:.0f}%)",
              fontsize=10, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


In [ ]:
# Clonal-redundancy check: MOB primary clusters are lineage proxies, so a compartment or host
# signal carried by one big cluster is one lineage counted many times, not an ecological pattern.
# Dereplication = one plasmid per cluster (seeded pick), matching the robustness check in
# reports/small_cryptic_methodology.md §3.
SEED = 20260727
vc = conj["mob_cluster"].value_counts()
derep = conj.sample(frac=1, random_state=SEED).drop_duplicates("mob_cluster")

print(f"{len(conj)} conjugative plasmids -> {conj['mob_cluster'].nunique()} MOB clusters "
      f"(mob_cluster called for {conj['mob_cluster'].notna().sum()}/{len(conj)})")
print(f"  {(vc == 1).sum()} singleton clusters | largest {vc.iloc[0]} "
      f"({vc.iloc[0] / len(conj) * 100:.0f}% of the set) | "
      f"top 5 clusters hold {vc.head(5).sum()} ({vc.head(5).sum() / len(conj) * 100:.0f}%)")

genus = conj["organism"].str.split().str[0]
acin = conj[genus.eq("Acinetobacter")]
print(f"\nAcinetobacter-genus plasmids: {len(acin)} in {acin['mob_cluster'].nunique()} clusters "
      f"-> {acin['mob_cluster'].value_counts().to_dict()}")

print("\ntop clusters:")
for cl in vc.head(8).index:
    s = conj[conj["mob_cluster"] == cl]
    print(f"  {cl}  n={len(s):>3}  host_range={s['mob_host_range'].mode().iat[0]:<18} "
          f"median {s['size_bp'].median():>6,.0f} bp  "
          f"compartments={s['compartment'].nunique()}")

# What survives dereplication.
def comp_pct(df):
    v = df["compartment"].value_counts().reindex(OH_ORDER, fill_value=0)
    v["(no compartment)"] = df["compartment"].isna().sum()
    return v / len(df) * 100

before, after = comp_pct(conj), comp_pct(derep)
print(f"\ncompartment mix, all {len(conj)} vs {len(derep)} dereplicated (% of set):")
print(pd.DataFrame({"all": before, "1 per cluster": after,
                    "shift": after - before}).round(1).to_string())

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))

topc = vc.head(10).iloc[::-1]
labels = [f"{cl}  ({conj.loc[conj['mob_cluster'] == cl, 'mob_host_range'].mode().iat[0]})"
          for cl in topc.index]
y = np.arange(len(topc))
axA.barh(y, topc.values, height=0.4, color=S1)
for i, v in enumerate(topc.values):
    axA.text(v + topc.max() * 0.02, i, f"{v}", va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=labels, xlim=(0, topc.max() * 1.12))
axA.set_xlabel("plasmids in cluster", fontsize=9, color=INK2)
axA.set_title(f"MOB cluster sizes — top 10 of {conj['mob_cluster'].nunique()}\n"
              f"{(vc == 1).sum()} clusters are singletons", fontsize=10, color=INK, loc="left")
tidy(axA)

y, h = np.arange(len(before)), 0.36
axB.barh(y + h / 2, before.values, height=h * 0.9, color=S1, label=f"all ({len(conj)})")
axB.barh(y - h / 2, after.values, height=h * 0.9, color=S2,
         label=f"1 per cluster ({len(derep)})")
axB.set(yticks=y, yticklabels=before.index)
axB.invert_yaxis()
axB.set_xlabel("% of set", fontsize=9, color=INK2)
axB.grid(axis="x", color=GRID, linewidth=1)
axB.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper right")
axB.set_title("One Health mix before vs after dereplication", fontsize=10, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


In [ ]:
# Replicons: small (< 20 kb) vs big (>= 20 kb), over the whole analysis base.
# Three independent replicon callers, so coverage is itself a finding, not just a caveat.
REPCOLS = ["pf_inc_types", "mob_rep_types", "plasann_replicons"]
groups = {"small (< 20 kb)": real[is_small], "big (>= 20 kb)": real[~is_small]}

cov = pd.DataFrame({
    g: {**{c: d[c].notna().mean() * 100 for c in REPCOLS},
        "any of the three": d[REPCOLS].notna().any(axis=1).mean() * 100}
    for g, d in groups.items()})
print("plasmids receiving a replicon call (% of group):")
print(cov.round(1).to_string(), f"\n  n: " +
      "  ".join(f"{g} {len(d):,}" for g, d in groups.items()))

# Replicon count. Report >=2 on both denominators: over the whole group the untyped plasmids
# sit in the 0 bin and drag it down, so "% of typed" is the like-for-like comparison.
cnt = pd.DataFrame({
    g: {"PlasmidFinder Inc, mean over group": d["pf_n_inc"].mean(),
        ">=2 Inc, % of group": (d["pf_n_inc"] >= 2).mean() * 100,
        ">=2 Inc, % of PF-typed": (d.loc[d["pf_inc_types"].notna(), "pf_n_inc"] >= 2).mean() * 100,
        "PlasAnn replicons, mean over group": d["plasann_n_replicons"].mean(),
        ">=2 PlasAnn, % of PlasAnn-typed":
            (d.loc[d["plasann_replicons"].notna(), "plasann_n_replicons"] >= 2).mean() * 100}
    for g, d in groups.items()})
print("\nreplicon multiplicity:")
print(cnt.round(2).to_string())

# Which replicon families, as % of the plasmids that got a PlasmidFinder call in that group.
def fam_pct(d):
    typed = d["pf_inc_families"].notna().sum()
    return (d["pf_inc_families"].dropna().str.split(";").explode().str.strip()
              .value_counts() / typed * 100)

fs, fb = fam_pct(groups["small (< 20 kb)"]), fam_pct(groups["big (>= 20 kb)"])
fam = (pd.DataFrame({"% of typed small": fs, "% of typed big": fb}).fillna(0)
         .loc[fs.head(10).index.union(fb.head(10).index)]
         .sort_values("% of typed small", ascending=False))
fam["diff (small - big)"] = fam["% of typed small"] - fam["% of typed big"]
print("\nPlasmidFinder Inc families — union of the top 10 of each group:")
print(fam.round(1).to_string())


In [ ]:
# Figure for cell 11. Colour follows the entity throughout: small = slot 1, big = slot 2,
# and the same two hues become the poles of the diverging panel.
SMALL_G, BIG_G = "small (< 20 kb)", "big (>= 20 kb)"

fig = plt.figure(figsize=(12, 12))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 2.1], hspace=0.35, wspace=0.25)
ax1, ax2, ax3 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])

def paired_columns(ax, frame, labels, title):
    x, w = np.arange(len(frame)), 0.3
    for k, (g, colour) in enumerate([(SMALL_G, S1), (BIG_G, S2)]):
        vals = frame[g].values
        ax.bar(x + (k - 0.5) * w, vals, width=w * 0.88, color=colour, label=g)
        for xi, v in zip(x + (k - 0.5) * w, vals):
            ax.text(xi, v + 2, f"{v:.1f}", ha="center", fontsize=8, color=INK2)
    ax.set(xticks=x, xticklabels=labels, yticks=[], ylim=(0, 100))
    ax.set_title(title, fontsize=10, color=INK, loc="left")
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper left")
    tidy(ax)

paired_columns(ax1, cov, ["Plasmid\nFinder", "mob_typer", "PlasAnn", "any of\nthe three"],
               "Plasmids receiving a replicon call (% of group)")
paired_columns(ax2, cnt.loc[[">=2 Inc, % of group", ">=2 Inc, % of PF-typed",
                            ">=2 PlasAnn, % of PlasAnn-typed"]],
               ["≥2 Inc\n(% of group)", "≥2 Inc\n(% of\nPF-typed)", "≥2 rep\n(% of\nPlasAnn-typed)"],
               "Multireplicon, by denominator and caller")

# Diverging bar: one axis, zero in the middle, warm/cool poles. Position already encodes the
# sign, so colour is reinforcement rather than the only channel.
fam_d = fam.sort_values("diff (small - big)")
y = np.arange(len(fam_d))
vals = fam_d["diff (small - big)"].values
ax3.barh(y, vals, height=0.4, color=[S1 if v > 0 else S2 for v in vals])
for i, v in enumerate(vals):
    ax3.text(v + (1.5 if v > 0 else -1.5), i, f"{v:+.1f}", va="center",
             ha="left" if v > 0 else "right", fontsize=8, color=INK2)
ax3.axvline(0, color=INK2, linewidth=1)
ax3.set(yticks=y, yticklabels=fam_d.index, xlim=(-60, 60))
ax3.set_xlabel("percentage-point difference, % of typed small − % of typed big",
               fontsize=9, color=INK2)
ax3.grid(axis="x", color=GRID, linewidth=1)
ax3.set_title("Inc family enrichment — union of the top 10 of each group\n"
              "left = enriched in big plasmids · right = enriched in small plasmids",
              fontsize=10, color=INK, loc="left")
tidy(ax3)

plt.show()


In [ ]:
# Replicon enrichment, small vs big, from mob_typer's mob_rep_types ONLY.
# Values are used exactly as provided — comma-separated, nothing stripped or collapsed, so
# rep_cluster_NNNN and ColRNAI_rep_cluster_NNNN stay as their own distinct types.
def rep_pct(d):
    typed = d["mob_rep_types"].notna().sum()
    return (d["mob_rep_types"].dropna().str.split(",").explode().str.strip()
              .value_counts() / typed * 100), typed

(rs, n_small), (rb, n_big) = rep_pct(groups[SMALL_G]), rep_pct(groups[BIG_G])
mrep = (pd.DataFrame({"% of typed small": rs, "% of typed big": rb}).fillna(0)
          .loc[rs.head(10).index.union(rb.head(10).index)])
mrep["diff (small - big)"] = mrep["% of typed small"] - mrep["% of typed big"]
mrep = mrep.sort_values("diff (small - big)")

print(f"mob_typer-typed: small {n_small:,} / {len(groups[SMALL_G]):,}  |  "
      f"big {n_big:,} / {len(groups[BIG_G]):,}")
print(f"distinct rep types: small {rs.size:,}  big {rb.size:,}\n")
print(mrep.sort_values("diff (small - big)", ascending=False).round(2).to_string())

fig, ax = plt.subplots(figsize=(11, 7))
y = np.arange(len(mrep))
vals = mrep["diff (small - big)"].values
lim = np.abs(vals).max() * 1.35                          # symmetric, so the two sides compare fairly
ax.barh(y, vals, height=0.4, color=[S1 if v > 0 else S2 for v in vals])
for i, v in enumerate(vals):
    ax.text(v + (lim * 0.015 if v > 0 else -lim * 0.015), i, f"{v:+.2f}", va="center",
            ha="left" if v > 0 else "right", fontsize=8, color=INK2)
ax.axvline(0, color=INK2, linewidth=1)
ax.set(yticks=y, yticklabels=mrep.index, xlim=(-lim, lim))
ax.set_xlabel("percentage-point difference, % of typed small − % of typed big",
              fontsize=9, color=INK2)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_title("mob_typer replicon-type enrichment — union of the top 10 of each group\n"
             f"types as provided, nothing stripped · denominators: "
             f"{n_small:,} typed small, {n_big:,} typed big\n"
             "left = enriched in big plasmids · right = enriched in small plasmids",
             fontsize=10, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()


In [ ]:
# Do small and big plasmids share MOB clusters (lineage proxies), or occupy separate ones?
# Singletons can never be mixed, so the mixing analysis runs on multi-member clusters only.
mc = real[real["mob_cluster"].notna()].copy()
mc["small"] = (mc["size_bp"] < SMALL_BP).values
codes, uniq = pd.factorize(mc["mob_cluster"])
csize = np.bincount(codes)
n_small = np.bincount(codes, weights=mc["small"].values).astype(int)
multi = csize >= 2
is_mixed = (n_small > 0) & (n_small < csize)

print(f"mob_cluster called for {len(mc):,}/{len(real):,} plasmids in {len(csize):,} clusters "
      f"({(csize == 1).sum():,} singletons, {multi.sum():,} multi-member)\n")
print("multi-member clusters:")
for lab, sel in [("mixed (both sizes)", is_mixed & multi),
                 ("small-only", (n_small == csize) & multi),
                 ("big-only", (n_small == 0) & multi)]:
    print(f"  {lab:<20s} {sel.sum():>6,} clusters ({sel.sum() / multi.sum() * 100:5.1f}%)  "
          f"{csize[sel].sum():>7,} plasmids")

# Is that more or less mixing than chance? Null = size labels shuffled across plasmids,
# holding cluster sizes and the overall small/big split fixed. One-sided (less mixing).
N_PERM = 1000
rng = np.random.default_rng(20260827)
lab = mc["small"].values.copy()
obs = int((is_mixed & multi).sum())
null = np.empty(N_PERM, dtype=int)
for i in range(N_PERM):
    rng.shuffle(lab)
    ns = np.bincount(codes, weights=lab).astype(int)
    null[i] = (((ns > 0) & (ns < csize)) & multi).sum()
p = (int((null <= obs).sum()) + 1) / (N_PERM + 1)
print(f"\npermutation test ({N_PERM} shuffles), mixed multi-member clusters:")
print(f"  observed {obs:,}  |  null {null.mean():,.0f} +/- {null.std():.0f} "
      f"(range {null.min():,}-{null.max():,})")
print(f"  obs/exp = {obs / null.mean():.3f}   one-sided p = {p:.4f}"
      f"{' (< 1/N_PERM)' if p <= 1 / (N_PERM + 1) else ''}")

# How much of plasmid size is explained by lineage? One-way random-effects ICC on log10(size).
y = np.log10(mc["size_bp"].values)
k, n = len(csize), len(y)
cmean = np.bincount(codes, weights=y) / csize
msb = (csize * (cmean - y.mean()) ** 2).sum() / (k - 1)
msw = ((y - cmean[codes]) ** 2).sum() / (n - k)
k0 = (n - (csize ** 2).sum() / n) / (k - 1)
print(f"\nICC of log10(size_bp) by cluster: {(msb - msw) / (msb + (k0 - 1) * msw):.3f}"
      f"   (fraction of size variance sitting between lineages)")

# Mixed clusters: genuinely balanced, or one size with a handful of the other?
minor = np.minimum(n_small, csize - n_small)[is_mixed & multi] / csize[is_mixed & multi]
print(f"\nwithin the {obs:,} mixed clusters, minority-size fraction:")
print(f"  quartiles {np.percentile(minor, [25, 50, 75]).round(3)}  |  "
      f"{(minor < 0.10).mean() * 100:.1f}% have a minority under 10%")

top = pd.DataFrame({"cluster": uniq[(is_mixed & multi)], "n": csize[is_mixed & multi],
                    "% small": n_small[is_mixed & multi] / csize[is_mixed & multi] * 100
                    }).nlargest(8, "n")
print("\nlargest mixed clusters:")
print(top.round(1).to_string(index=False))


In [ ]:
# Figure for cell 14. small = slot 1 and big = slot 2 as everywhere else; "mixed" takes slot 3
# and carries that hue through panels B and C, which are both about the mixed clusters.
MIXED_C = S3
icc = (msb - msw) / (msb + (k0 - 1) * msw)          # msb/msw/k0 computed in the previous cell

fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(2, 2, height_ratios=[0.55, 2], hspace=0.5, wspace=0.22)
axA = fig.add_subplot(gs[0, :])
axB, axC = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])

# A — composition of the multi-member clusters, ordered all-small -> mixed -> all-big.
parts = [("small-only", ((n_small == csize) & multi).sum(), S1, "white"),
         ("mixed", int((is_mixed & multi).sum()), MIXED_C, INK),
         ("big-only", ((n_small == 0) & multi).sum(), S2, "white")]
tot, left = multi.sum(), 0.0
for label, v, colour, lab_ink in parts:
    axA.barh([0], [v / tot * 100], left=left, height=0.42, color=colour,
             edgecolor="white", linewidth=1.5)
    axA.text(left + v / tot * 50, 0, f"{label}\n{v:,} ({v / tot * 100:.1f}%)",
             ha="center", va="center", fontsize=9, color=lab_ink)
    left += v / tot * 100
axA.set(xlim=(0, 100), xticks=[0, 25, 50, 75, 100], yticks=[])
axA.set_xlabel("% of multi-member clusters", fontsize=9, color=INK2)
axA.set_title(f"MOB clusters by size composition  ({tot:,} clusters with >=2 members; "
              f"{(csize == 1).sum():,} singletons excluded)", fontsize=11, color=INK, loc="left")
tidy(axA)

# B — observed vs chance. The null's spread is 0.4% of its mean, so it is a point, not a
# distribution: two bars beat a histogram here.
bars = [("observed", obs, MIXED_C), (f"expected by chance\n(±{null.std():.0f}, "
                                     f"{N_PERM:,} shuffles)", null.mean(), NEUT)]
yb = np.arange(2)
axB.barh(yb, [v for _, v, _ in bars], height=0.3, color=[c for _, _, c in bars])
for i, (lab, v, _) in enumerate(bars):
    axB.text(v + null.mean() * 0.02, i, f"{v:,.0f}", va="center", fontsize=9, color=INK2)
axB.set(yticks=yb, yticklabels=[lab for lab, _, _ in bars],
        xlim=(0, null.mean() * 1.18), ylim=(1.9, -0.9))   # already top-down; no invert
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel("mixed clusters", fontsize=9, color=INK2)
axB.set_title(f"Size mixing vs chance\nobs/exp = {obs / null.mean():.3f} — lineages are far "
              f"more size-pure than random", fontsize=10, color=INK, loc="left")
tidy(axB)

# C — and the mixed ones are mostly lopsided, not balanced.
axC.hist(minor, bins=25, range=(0, 0.5), color=MIXED_C, edgecolor="white", linewidth=0.5)
axC.axvline(np.median(minor), color=INK, linewidth=2)
axC.text(np.median(minor) + 0.012, axC.get_ylim()[1] * 0.95,
         f"median {np.median(minor):.2f}", fontsize=9, color=INK, va="top")
axC.set(xlim=(0, 0.5), yticks=[])
axC.set_xlabel("minority-size fraction within the cluster", fontsize=9, color=INK2)
axC.text(1.0, -0.155, "spikes at 1/2 and 1/3 are small clusters (n=2, n=3), where the "
         "fraction can only take a few values", fontsize=8, color=INK2, ha="right",
         transform=axC.transAxes)
axC.set_title(f"How balanced are the {obs:,} mixed clusters?\n"
              f"{(minor < 0.10).mean() * 100:.0f}% have a minority under 10% — "
              f"0.5 would be an even split", fontsize=10, color=INK, loc="left")
tidy(axC)

fig.suptitle(f"Do small and big plasmids share MOB lineages?   "
             f"ICC of log10(size) by cluster = {icc:.3f}", fontsize=12, color=INK,
             x=0.008, ha="left", y=0.99)
plt.show()


In [ ]:
# What is on the payload-free ("cryptic") small plasmids — the 71,414 from cell 2.
# ~5 min: scans all 1,072 PlasAnn shards. usecols keeps this to ~70 MB.
cryptic_small = real[is_small & is_cryptic]
print(f"cryptic small: {len(cryptic_small):,}  "
      f"({len(cryptic_small) / is_small.sum() * 100:.1f}% of small, "
      f"{len(cryptic_small) / len(real) * 100:.1f}% of the analysis base)\n")

print("physical (quartiles):")
print(cryptic_small[["size_bp", "gc_percent", "plasann_n_cds"]]
      .describe().loc[["25%", "50%", "75%"]].round(1).to_string())
print(f"\nmobility: {cryptic_small['mob_mobility'].value_counts(dropna=False).to_dict()}")
print(f"topology: {cryptic_small['topology'].value_counts(dropna=False).head(3).to_dict()}")

cids = set(cryptic_small["plasmid_id"])
parts = []
for f in sorted(glob.glob(str(ROOT / "data" / "plasann_run" / "annot" / "*shard_*.tsv.gz"))):
    d = pd.read_csv(f, sep="\t", usecols=["plasmid_id", "Gene Name", "Category"],
                    low_memory=False)
    d = d[d["plasmid_id"].isin(cids)]
    if len(d):
        parts.append(d)
cg = pd.concat(parts, ignore_index=True)
ccov = cg["plasmid_id"].nunique()
print(f"\nper-gene coverage: {ccov:,}/{len(cids):,} ({ccov / len(cids) * 100:.1f}%), "
      f"{len(cg):,} CDS rows\n")

vc2 = cg["Category"].value_counts()
ccat = pd.DataFrame({"CDS": vc2, "% of CDS": vc2 / len(cg) * 100,
                     "plasmids": cg.groupby("Category")["plasmid_id"].nunique()})
ccat["% of plasmids"] = ccat["plasmids"] / ccov * 100
print(ccat.sort_values("plasmids", ascending=False).round(1).to_string())

c_orf = cg["Gene Name"].eq("ORF")
c_dark = c_orf.groupby(cg["plasmid_id"]).mean()
print(f"\nunannotatable (ORF): {c_orf.mean() * 100:.1f}% of CDS pooled | "
      f"median per-plasmid dark fraction {c_dark.median():.2f} | "
      f"{(c_dark == 1).sum():,} plasmids ({(c_dark == 1).mean() * 100:.1f}%) entirely dark")

named = cg[~c_orf]
print(f"\ntop named genes ({named['Gene Name'].nunique():,} distinct, {len(named):,} CDS):")
print(named.groupby("Gene Name")["plasmid_id"].nunique()
           .sort_values(ascending=False).head(20).to_string())

# The payload filter used CARD for AMR, so PlasAnn's own AMR calls are not excluded by it.
leak = int(ccat.loc["Antibiotic Resistance", "plasmids"])
print(f"\nNOTE: {leak:,} of these 'payload-free' plasmids carry a PlasAnn Antibiotic-Resistance "
      f"CDS.\n  Not a contradiction — cell 2 defined AMR by card_n_arg (CARD/RGI, Perfect+Strict),"
      f"\n  not plasann_n_amr. Adding plasann_n_amr to PAYLOAD would drop them.")


In [ ]:
# Size distribution of the cryptic small plasmids. One series -> one colour, no legend.
sz = cryptic_small["size_bp"].values
q1, med, q3 = np.percentile(sz, [25, 50, 75])

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.hist(sz, bins=80, range=(0, SMALL_BP), color=S1, edgecolor="white", linewidth=0.4)
ax.axvspan(q1, q3, color=S1, alpha=0.10, zorder=0)         # IQR as a wash, not a saturated block
ax.axvline(med, color=INK, linewidth=2)
ax.text(med + 250, ax.get_ylim()[1] * 0.94,
        f"median {med:,.0f} bp\nIQR {q1:,.0f}–{q3:,.0f}", fontsize=9, color=INK, va="top")
ax.set(xlim=(0, SMALL_BP), yticks=[])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.set_xlabel("size (bp)", fontsize=9, color=INK2)
ax.set_title(f"Size distribution — payload-free small plasmids  (n = {len(sz):,})\n"
             f"min {sz.min():,} bp · max {sz.max():,} bp · "
             f"{(sz < 10_000).mean() * 100:.0f}% are under 10 kb",
             fontsize=11, color=INK, loc="left")
tidy(ax)
plt.tight_layout()
plt.show()


In [ ]:
# Replicon types on the cryptic small plasmids — mob_typer's mob_rep_types only, as provided.
# The no-call group is ~19x the commonest single type, so it gets its own panel rather than
# sitting on the same axis where it would flatten every type bar.
c_typed = cryptic_small["mob_rep_types"].notna()
c_rep = (cryptic_small["mob_rep_types"].dropna().str.split(",").explode().str.strip())
c_counts = c_rep.value_counts()

print(f"cryptic small: {len(cryptic_small):,} | with a mob_rep_types call {c_typed.sum():,} "
      f"({c_typed.mean() * 100:.1f}%) | no call {(~c_typed).sum():,} "
      f"({(~c_typed).mean() * 100:.1f}%)")
print(f"distinct rep types among the typed: {c_counts.size:,}")
print(f"top 15 cover {c_counts.head(15).sum():,} calls "
      f"({c_counts.head(15).sum() / c_counts.sum() * 100:.1f}% of all calls)\n")
print(c_counts.head(15).to_string())

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5.6),
                               gridspec_kw={"width_ratios": [1, 1.6]})

# A — how many get typed at all. Grey = absence of a call, as everywhere else in this notebook.
covr = [("typed", int(c_typed.sum()), S1), ("no replicon call", int((~c_typed).sum()), NEUT)]
yA = np.arange(2)
axA.barh(yA, [v for _, v, _ in covr], height=0.3, color=[c for _, _, c in covr])
for i, (_, v, _) in enumerate(covr):
    axA.text(v + len(cryptic_small) * 0.02, i, f"{v:,}\n({v / len(cryptic_small) * 100:.1f}%)",
             va="center", fontsize=9, color=INK2)
axA.set(yticks=yA, yticklabels=[lab for lab, _, _ in covr],
        xlim=(0, len(cryptic_small) * 1.25), ylim=(1.9, -0.9),
        xticks=[0, 20_000, 40_000, 60_000])
axA.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axA.set_xlabel("plasmids", fontsize=9, color=INK2)
axA.set_title(f"Replicon typing coverage  (n = {len(cryptic_small):,})\n"
              f"mob_typer only, per the standing rule", fontsize=10, color=INK, loc="left")
tidy(axA)

# B — and among the typed minority, which types.
topr = c_counts.head(15).iloc[::-1]
yB = np.arange(len(topr))
axB.barh(yB, topr.values, height=0.4, color=S1)
for i, v in enumerate(topr.values):
    axB.text(v + topr.max() * 0.015, i, f"{v:,}", va="center", fontsize=8, color=INK2)
axB.set(yticks=yB, yticklabels=topr.index, xlim=(0, topr.max() * 1.15))
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel(f"plasmids (of the {c_typed.sum():,} typed)", fontsize=9, color=INK2)
axB.set_title(f"Top 15 replicon types of {c_counts.size:,} distinct\n"
              f"a long tail, no dominant type", fontsize=10, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


In [ ]:
# Full annotated inventory of the cryptic small plasmids: every PlasAnn gene call, ranked
# inside its own category. Reuses cg / cryptic_small from cell 16, so no re-scan of the shards.
# "PLasmid Maintenance..." (4 CDS) is a case typo of an existing category — merged into it.
cgn = cg.copy()
cgn["Category"] = cgn["Category"].str.replace(r"^PLasmid", "Plasmid", regex=True)
COV = cgn["plasmid_id"].nunique()

cnamed = cgn[cgn["Gene Name"].ne("ORF")]
gcount = cnamed.groupby("Gene Name")["plasmid_id"].nunique()
cum = cnamed["Gene Name"].value_counts().cumsum() / len(cnamed)
print(f"annotated (non-ORF) CDS: {len(cnamed):,} / {len(cgn):,} "
      f"({len(cnamed) / len(cgn) * 100:.1f}%), on {cnamed['plasmid_id'].nunique():,} "
      f"of {COV:,} plasmids with gene data")
print(f"distinct gene calls: {gcount.size:,}  |  {(gcount == 1).sum():,} "
      f"({(gcount == 1).mean() * 100:.0f}%) appear on exactly one plasmid  |  "
      f"{(cum <= 0.90).sum() + 1} genes cover 90% of annotated CDS\n")

INV = (cnamed.groupby("Category")
       .agg(CDS=("Gene Name", "size"), genes=("Gene Name", "nunique"),
            plasmids=("plasmid_id", "nunique"))
       .sort_values("plasmids", ascending=False))
INV["% of plasmids"] = INV["plasmids"] / COV * 100
print(INV.round(1).to_string())

# Every category, its top 10 genes by how many plasmids carry them.
for c in INV.index:
    t = (cnamed[cnamed["Category"].eq(c)].groupby("Gene Name")["plasmid_id"]
         .nunique().sort_values(ascending=False))
    print(f"\n{c}  —  {INV.loc[c, 'plasmids']:,} plasmids, {t.size:,} distinct genes, "
          f"top gene on {t.iloc[0] / INV.loc[c, 'plasmids'] * 100:.0f}% of them")
    print("  " + ", ".join(f"{k} {v:,}" for k, v in t.head(10).items()))


In [ ]:
# Figure for the inventory. One small multiple per category, top 6 genes each, own x scale
# because the categories differ by two orders of magnitude in size. Origin of Replication and
# Origin of Transfer hold a single gene each (oriV, oriT) so they carry no panel.
SHORT = {"Plasmid Maintenance, Replication and Regulation": "Maintenance / replication / regulation",
         "Non coding RNA/Regulatory elements": "ncRNA / regulatory elements",
         "Non-conjugative DNA mobility": "Non-conjugative DNA mobility",
         "Toxin-Antitoxin System": "Toxin-antitoxin systems"}
panels = [c for c in INV.index if INV.loc[c, "genes"] > 1]

fig, axes = plt.subplots(4, 3, figsize=(15, 13))
for ax, c in zip(axes.ravel(), panels):
    t = (cnamed[cnamed["Category"].eq(c)].groupby("Gene Name")["plasmid_id"]
         .nunique().sort_values(ascending=False).head(6).iloc[::-1])
    y = np.arange(len(t))
    ax.barh(y, t.values, height=0.42, color=S1)
    for i, v in enumerate(t.values):
        ax.text(v + t.max() * 0.02, i, f"{v:,}", va="center", fontsize=8, color=INK2)
    ax.set(yticks=y, yticklabels=t.index, xlim=(0, t.max() * 1.22), xticks=[])
    ax.set_title(f"{SHORT.get(c, c)}\n{INV.loc[c, 'plasmids']:,} plasmids · "
                 f"{INV.loc[c, 'genes']:,} distinct genes", fontsize=10, color=INK, loc="left")
    tidy(ax)
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")

fig.suptitle(f"What PlasAnn actually calls on the payload-free small plasmids\n"
             f"top 6 genes per category, by plasmids carrying them (of {COV:,} with gene data) · "
             f"single-gene categories not shown: oriV {INV.loc['Origin of Replication', 'plasmids']:,}"
             f" · oriT {INV.loc['Origin of Transfer', 'plasmids']:,}\n"
             f"each panel has its own scale — bars are comparable within a panel, not across them",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.0)
plt.tight_layout(rect=(0, 0, 1, 0.945))
plt.show()


In [ ]:
# Per-plasmid architecture: which functional modules co-occur on one replicon.
# Categories are collapsed into modules so the combination space stays readable (2^5, not 2^13).
# The mapping is a choice, stated here rather than buried: ncRNA sits in "other" even though
# RNAI is replication control, because the category also holds tRNAs and unrelated sRNAs.
MODULE = {
    "Origin of Replication": "replication", "Replicon": "replication",
    "Plasmid Maintenance, Replication and Regulation": "replication",
    "Conjugation": "transfer", "Origin of Transfer": "transfer",
    "Mobile Element": "recombination", "Non-conjugative DNA mobility": "recombination",
    "Toxin-Antitoxin System": "maintenance (TA)",
    "Non coding RNA/Regulatory elements": "other", "Other": "other", "Metabolism": "other",
    "Stress Response": "other", "Antibiotic Resistance": "other",
}
MODS = ["replication", "transfer", "recombination", "maintenance (TA)", "other"]

# Denominator is the plasmids PlasAnn returned genes for; a plasmid with only ORFs is present
# here with every module False, which is exactly the "fully dark" row of the table.
allids = pd.Index(cgn["plasmid_id"].unique())
mod = cnamed.assign(module=cnamed["Category"].map(MODULE))
pm = (pd.crosstab(mod["plasmid_id"], mod["module"]) > 0).reindex(allids, fill_value=False)[MODS]

print(f"{len(pm):,} plasmids with PlasAnn gene data "
      f"({len(cryptic_small) - len(pm):,} of the {len(cryptic_small):,} have none)\n")
print("module prevalence (% of plasmids):")
print((pm.mean() * 100).round(1).to_string())

nmod = pm.sum(axis=1)
print("\nmodules per plasmid:")
nm = pd.DataFrame({"plasmids": nmod.value_counts().sort_index()})
nm["%"] = nm["plasmids"] / len(pm) * 100
nm["median bp"] = (cryptic_small.set_index("plasmid_id")["size_bp"]
                   .reindex(pm.index).groupby(nmod).median())
nm["median CDS"] = cgn.groupby("plasmid_id").size().reindex(pm.index).groupby(nmod).median()
print(nm.round(1).to_string())

combo = pm.apply(lambda r: " + ".join([m for m in MODS if r[m]]) or "none (fully dark)", axis=1)
vc3 = combo.value_counts()
print(f"\nmodule combinations — top 15 of the {vc3.size} that occur:")
print(pd.DataFrame({"plasmids": vc3, "%": vc3 / len(pm) * 100}).head(15).round(1).to_string())

# Collapsed to the two modules that define a plasmid's minimal lifestyle, plus a catch-all.
core = pm[["replication", "transfer"]].copy()
core["anything else"] = pm[["recombination", "maintenance (TA)", "other"]].any(axis=1)
c3 = core.apply(lambda r: " + ".join([m for m in core.columns if r[m]]) or "none (fully dark)",
                axis=1)
vc4 = c3.value_counts()
print("\ncollapsed to replication x transfer x anything-else:")
print(pd.DataFrame({"plasmids": vc4, "%": vc4 / len(pm) * 100}).round(1).to_string())

# Pairwise co-occurrence at category level. CAVEAT: nearly every value below is positive, because
# the shared driver is annotation depth — a plasmid with more annotated CDS is likelier to hit
# any category. Read the ranking, not the absolute values.
CATS = [c for c in INV.index if INV.loc[c, "plasmids"] >= 3000]
pc = (pd.crosstab(cnamed["plasmid_id"], cnamed["Category"]) > 0).reindex(allids,
                                                                        fill_value=False)[CATS]
p1 = pc.mean().values
obs = (pc.astype(int).T @ pc.astype(int)).values / len(pc)
lr = np.log2(obs / np.outer(p1, p1))
np.fill_diagonal(lr, np.nan)
print("\nlog2 observed/expected co-occurrence (see the depth caveat above):")
print(pd.DataFrame(lr, index=[c[:26] for c in CATS],
                   columns=[c[:10] for c in CATS]).round(2).to_string())


In [ ]:
# Figure for the architecture. Panel A is an UpSet-style plot: the dot matrix says which modules
# are present, the bar says how many plasmids have exactly that combination and nothing else.
# The empty row is the fully-dark set, so grey = absence of any annotated module, as elsewhere.
TOPK = 12
top_combo = vc3.head(TOPK)
present = [[] if k.startswith("none") else k.split(" + ") for k in top_combo.index]

fig = plt.figure(figsize=(13, 10))
gs = fig.add_gridspec(2, 2, height_ratios=[2.3, 1], width_ratios=[0.5, 1],
                      hspace=0.42, wspace=0.02)
axM, axA, axB = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])

y = np.arange(TOPK)
# Dot matrix. Filled = module present; the line joins the members of one combination.
for i, mods in enumerate(present):
    xs = [MODS.index(m) for m in mods]
    axM.scatter(range(len(MODS)), [i] * len(MODS), s=52, color=GRID, zorder=1)
    if xs:
        axM.plot([min(xs), max(xs)], [i, i], color=S1, linewidth=1.6, zorder=2)
        axM.scatter(xs, [i] * len(xs), s=52, color=S1, zorder=3)
axM.set(xlim=(-0.6, len(MODS) - 0.4), ylim=(TOPK - 0.5, -0.5),
        xticks=range(len(MODS)), yticks=[])
axM.set_xticklabels(MODS, rotation=40, ha="right", fontsize=9)
for s in ("top", "right", "left", "bottom"):
    axM.spines[s].set_visible(False)
axM.tick_params(colors=INK2, length=0)

vals = top_combo.values
axA.barh(y, vals, height=0.42,
         color=[NEUT if k.startswith("none") else S1 for k in top_combo.index])
for i, v in enumerate(vals):
    axA.text(v + vals.max() * 0.015, i, f"{v:,}  ({v / len(pm) * 100:.1f}%)",
             va="center", fontsize=9, color=INK2)
axA.set(yticks=[], ylim=(TOPK - 0.5, -0.5), xlim=(0, vals.max() * 1.26),
        xticks=[0, 10_000, 20_000, 30_000])
axA.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axA.set_xlabel(f"plasmids with exactly this combination (of {len(pm):,})",
               fontsize=9, color=INK2)
axA.set_title(f"Module architecture — top {TOPK} of {vc3.size} combinations "
              f"({top_combo.sum() / len(pm) * 100:.0f}% of the set)",
              fontsize=11, color=INK, loc="left")
tidy(axA)

# B — how many modules a single plasmid carries at all.
nb = nm["plasmids"]
xb = np.arange(len(nb))
axB.bar(xb, nb.values, width=0.42, color=[NEUT if k == 0 else S1 for k in nb.index])
for i, (k, v) in enumerate(nb.items()):
    axB.text(i, v + nb.max() * 0.02, f"{v:,}\n{v / len(pm) * 100:.1f}%",
             ha="center", fontsize=9, color=INK2)
axB.set(xticks=xb, xticklabels=nb.index, yticks=[], ylim=(0, nb.max() * 1.22))
axB.set_xlabel("annotated modules on the plasmid", fontsize=9, color=INK2)
axB.set_title(f"Nearly half carry no annotated module at all — "
              f"{nb.loc[0]:,} plasmids are ORFs end to end",
              fontsize=11, color=INK, loc="left")
tidy(axB)

fig.suptitle("Per-plasmid architecture of the payload-free small plasmids", fontsize=12,
             color=INK, x=0.008, ha="left", y=0.98)
plt.show()


In [ ]:
# How many dark ORFs sit on each cryptic small plasmid.
# Two independent routes to the same quantity, computed side by side rather than trusting one:
#   TSV   — cg from cell 16, rows where Gene Name == "ORF" (same source as cells 16/19)
#   FASTA — headers of data/dark_orf_run/dark_orfs.faa, the dark proteome that was actually
#           clustered and Pfam-searched, so it is the count the downstream results rest on.
# Header grammar: PLASMID|cds_index in dark_orfs.faa, PLASMID|cds_index|{D,N} in all_cds.faa,
# where D/N marks dark vs named — so all_cds.faa supplies the per-plasmid CDS denominator.
from collections import Counter

from scipy import stats

DOR = ROOT / "data" / "dark_orf_run"


def hdr_counts(path, nfields):
    c = Counter()
    with open(path) as fh:
        for line in fh:
            if line[0] == ">":
                c[line[1:].rstrip().rsplit("|", nfields)[0]] += 1
    return c


DK = cryptic_small[["plasmid_id", "size_bp"]].set_index("plasmid_id").copy()
DK["cds"] = pd.Series(hdr_counts(DOR / "all_cds.faa", 2)).reindex(DK.index)
DK["dark"] = pd.Series(hdr_counts(DOR / "dark_orfs.faa", 1)).reindex(DK.index)

# NaN cds means the GenBank output held no CDS for that plasmid — an upstream PlasAnn coverage
# gap, not a plasmid without genes (their plasann_n_cds is non-zero). Those cannot supply a zero,
# so they are dropped rather than folded in as "0 dark ORFs".
have = DK["cds"].notna()
DK.loc[have, "dark"] = DK.loc[have, "dark"].fillna(0)
D = DK[have].astype({"cds": int, "dark": int})

print(f"cryptic small plasmids:            {len(DK):>7,}")
print(f"  PlasAnn returned CDS:            {len(D):>7,}  ({len(D) / len(DK) * 100:.1f}%)"
      f"   <- denominator throughout")
print(f"  no CDS in GenBank output:        {(~have).sum():>7,}  ({(~have).mean() * 100:.1f}%)"
      f"   (median {DK.loc[~have, 'size_bp'].median():,.0f} bp; excluded, not counted as zero)")
print(f"  carrying >= 1 dark ORF:          {(D['dark'] > 0).sum():>7,}  "
      f"({(D['dark'] > 0).mean() * 100:.1f}% of the {len(D):,})")
print(f"  every CDS named, no dark ORF:    {(D['dark'] == 0).sum():>7,}  "
      f"({(D['dark'] == 0).mean() * 100:.1f}%)")

q = D["dark"].quantile([0.25, 0.5, 0.75, 0.9, 0.99])
print(f"\ndark ORFs per plasmid (all {len(D):,}): "
      f"median {q[0.5]:.0f}  IQR {q[0.25]:.0f}-{q[0.75]:.0f}  "
      f"p90 {q[0.9]:.0f}  p99 {q[0.99]:.0f}  max {D['dark'].max()}  "
      f"mean {D['dark'].mean():.2f}  total {D['dark'].sum():,}")
qn = D.loc[D["dark"] > 0, "dark"].quantile([0.25, 0.5, 0.75])
print(f"dark ORFs per plasmid (the {(D['dark'] > 0).sum():,} that carry any): "
      f"median {qn[0.5]:.0f}  IQR {qn[0.25]:.0f}-{qn[0.75]:.0f}")

tab = D["dark"].value_counts().sort_index()
grp = pd.Series({"0": tab.get(0, 0), **{str(k): tab.get(k, 0) for k in range(1, 11)},
                 "11-20": tab.loc[11:20].sum(), "21+": tab.loc[21:].sum()})
held = ([0] + [k * tab.get(k, 0) for k in range(1, 11)]
        + [sum(k * tab.get(k, 0) for k in range(11, 21)),
           sum(k * v for k, v in tab.items() if k >= 21)])
print("\nplasmids by dark ORF count:")
print(pd.DataFrame({"plasmids": grp, "% of plasmids": grp / len(D) * 100,
                    "dark ORFs held": held,
                    "% of dark ORFs": np.array(held) / D["dark"].sum() * 100}).round(1).to_string())

# A plasmid is "fully dark" when PlasAnn named none of its CDS. That is the set cells 21-22
# called "none (fully dark)"; here it gets a size in ORFs, not just a share of plasmids.
full = D["dark"].eq(D["cds"]) & D["cds"].gt(0)
print(f"\nfully dark (every CDS an ORF): {full.sum():,} plasmids ({full.mean() * 100:.1f}%), "
      f"holding {D.loc[full, 'dark'].sum():,} dark ORFs "
      f"({D.loc[full, 'dark'].sum() / D['dark'].sum() * 100:.1f}% of the dark proteome); "
      f"median {D.loc[full, 'dark'].median():.0f} ORFs on {D.loc[full, 'size_bp'].median():,.0f} bp")

# The count is largely a size statement, so give the rate alongside it.
rate = D["dark"] / (D["size_bp"] / 1000)
print(f"\ndark ORFs per kb: median {rate.median():.2f}  "
      f"IQR {rate.quantile(.25):.2f}-{rate.quantile(.75):.2f}   "
      f"(size vs count: Spearman rho = {stats.spearmanr(D['size_bp'], D['dark']).statistic:.3f})")

BINS = [0, 2_000, 4_000, 6_000, 10_000, 15_000, 20_000]
LAB = ["< 2 kb", "2-4 kb", "4-6 kb", "6-10 kb", "10-15 kb", "15-20 kb"]
D["szbin"] = pd.cut(D["size_bp"], BINS, labels=LAB)
SZ = D.groupby("szbin", observed=True).apply(
    lambda t: pd.Series({"plasmids": len(t), "median dark": t["dark"].median(),
                         "q1": t["dark"].quantile(.25), "q3": t["dark"].quantile(.75),
                         "median CDS": t["cds"].median(),
                         "dark/kb": (t["dark"] / (t["size_bp"] / 1000)).median(),
                         "% fully dark": (t["dark"] == t["cds"]).mean() * 100}),
    include_groups=False)
print("\nby plasmid size:")
print(SZ.round(2).to_string())

# Cross-check. The two routes are different files written by different scripts (PlasAnn's TSV
# shards vs the GenBank protein extraction), so agreement is worth printing, not assuming.
tsv_n = cg.loc[cg["Gene Name"].eq("ORF")].groupby("plasmid_id").size()
gap = pd.Index(cg["plasmid_id"].unique()).difference(D.index).size
both = D.index.intersection(tsv_n.index)
agree = (tsv_n.reindex(both) == D.loc[both, "dark"]).mean()
print(f"\ncross-check TSV vs FASTA: {len(both):,} plasmids in both, "
      f"{agree * 100:.2f}% identical counts | totals {tsv_n.sum():,} vs {D['dark'].sum():,} | "
      f"{gap} plasmids have PlasAnn gene rows but no CDS in the GenBank output")

In [ ]:
# Figure for the per-plasmid dark ORF count. Colour carries one meaning each, across all three
# panels: slot 1 = dark ORFs, slot 2 = all CDS, grey = a plasmid with no dark ORF at all.
XMAX = 25
bars = tab.reindex(range(XMAX + 1), fill_value=0)
tail = int(D["dark"].gt(XMAX).sum())
med = D["dark"].median()

fig = plt.figure(figsize=(15, 4.8))
gs = fig.add_gridspec(1, 3, width_ratios=[1.55, 1, 1], wspace=0.28)
axA, axB, axC = (fig.add_subplot(gs[0, i]) for i in range(3))

# A — the distribution itself, one bar per integer count.
axA.bar(bars.index, bars.values, width=0.62,
        color=[NEUT if k == 0 else S1 for k in bars.index])
axA.axvline(med, color=INK, linewidth=1.6)
axA.text(med + 0.4, bars.max() * 0.95, f"median {med:.0f}", fontsize=9, color=INK, va="top")
axA.text(XMAX - 0.2, bars.max() * 0.60,
         f"{tail:,} plasmids carry\nmore than {XMAX} (max {D['dark'].max()})",
         fontsize=8.5, color=INK2, ha="right", va="top")
axA.set(xlim=(-0.8, XMAX + 0.8), xticks=range(0, XMAX + 1, 5), yticks=[],
        ylim=(0, bars.max() * 1.16))
axA.set_xlabel("dark ORFs on the plasmid", fontsize=9, color=INK2)
axA.set_title(f"A — {(D['dark'] > 0).sum():,} of {len(D):,} carry at least one\n"
              f"grey: {int(bars.loc[0]):,} have every CDS named",
              fontsize=10, color=INK, loc="left")
tidy(axA)

# B — the count is largely a size statement. Both series share an axis: both are CDS counts.
mid = np.array([1_000, 3_000, 5_000, 8_000, 12_500, 17_500])
axB.fill_between(mid, SZ["q1"].values, SZ["q3"].values, color=S1, alpha=0.16, linewidth=0)
axB.plot(mid, SZ["median CDS"].values, color=S2, linewidth=1.8, marker="o", markersize=4)
axB.plot(mid, SZ["median dark"].values, color=S1, linewidth=2.2, marker="o", markersize=5)
axB.text(mid[3] - 400, SZ["median CDS"].values[3] + 1.2, "all CDS", fontsize=9, color=S2,
         ha="right")
axB.text(mid[3] + 400, SZ["median dark"].values[3] - 1.6, "dark ORFs", fontsize=9, color=S1,
         ha="left", va="top")
axB.set(xlim=(0, SMALL_BP), xticks=[0, 5_000, 10_000, 15_000, 20_000],
        ylim=(0, SZ["median CDS"].max() * 1.18))
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel("plasmid size (bp)", fontsize=9, color=INK2)
axB.set_ylabel("CDS per plasmid (median, IQR shaded)", fontsize=9, color=INK2)
axB.set_title(f"B — the count is a size statement\n"
              f"rate flat: {rate.median():.2f} dark ORFs per kb",
              fontsize=10, color=INK, loc="left")
tidy(axB)

# C — banded dark fraction. The two ends are the informative rows: nothing dark, everything dark.
frac = D["dark"] / D["cds"]
ORD = ["none", "<25%", "25-50%", "50-75%", "75-<100%", "all"]
band = pd.cut(frac, [-0.001, 0.0, 0.25, 0.5, 0.75, 0.999, 1.0],
              labels=ORD).value_counts().reindex(ORD)
yb = np.arange(len(band))[::-1]
axC.barh(yb, band.values, height=0.5,
         color=[NEUT if k == "none" else S1 for k in band.index])
for y, v in zip(yb, band.values):
    axC.text(v + band.max() * 0.02, y, f"{v:,}  ({v / len(D) * 100:.1f}%)",
             va="center", fontsize=8.5, color=INK2)
axC.set(yticks=yb, yticklabels=band.index, xticks=[], xlim=(0, band.max() * 1.34))
axC.set_xlabel("plasmids", fontsize=9, color=INK2)
axC.set_title(f"C — how much of the plasmid is dark\n"
              f"{full.mean() * 100:.1f}% dark end to end "
              f"({D.loc[full, 'dark'].sum() / D['dark'].sum() * 100:.0f}% of dark ORFs)",
              fontsize=10, color=INK, loc="left")
tidy(axC)

fig.suptitle(f"Dark ORFs per cryptic small plasmid  —  {D['dark'].sum():,} dark ORFs over "
             f"{len(D):,} plasmids, median {med:.0f} each",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.04)
plt.show()